In [3]:
pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [24]:
from kaggle_secrets import UserSecretsClient
import os
from google import genai

# Kaggle ke Secrets se key uthane ka tareeqa
user_secrets = UserSecretsClient()
gemini_key = user_secrets.get_secret("AI Course Lab")

# Client initialize karein aur key pass kar dein
client = genai.Client(api_key=gemini_key)

print("Gemini client initialized via Kaggle Secrets!")

Gemini client initialized via Kaggle Secrets!


In [25]:
# Task 1.2: Make First API Call (Gemini Version)

# Simple completion call
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Say hello and introduce yourself!',
    config={
        'max_output_tokens': 150,  # OpenAI ka max_tokens yahan max_output_tokens ban jata hai
        'temperature': 0.7
    }
)

# Extract response
message = response.text  # Gemini mein response.choices[0]... likhne ki zaroorat nahi hoti, direct .text hota hai
print(message)

Hello! I'm a large language model, trained by Google. It's nice to meet you!


In [27]:
import os
from google import genai
from google.genai import types

# --- CONVERSATION LOOP FUNCTION (Task 2.1 & 2.2) ---
def start_gemini_bot(system_instruction=None):
    """
    Starts a stateful multi-turn conversation loop using the Gemini client.
    Gemini's chats.create handles conversation history automatically.
    """
    # Using the correct modern Gemini client instance and setting up the system prompt
    chat_session = client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.7
        ) if system_instruction else None
    )
    
    print('Chatbot ready! Type "quit" to exit.\n')
    
    while True:
        user_input = input('You: ')
        if user_input.lower() == 'quit':
            print('Goodbye!')
            break
            
        # Send message to the session (keeps track of dialogue history context)
        response = chat_session.send_message(user_input)
        print(f'Assistant: {response.text}\n')


# --- TASK 3.1: HR ASSISTANT BOT ---
hr_system_prompt = """You are an HR assistant for a technology company.
Company policies:
- Vacation: 15 days per year
- Sick leave: Unlimited (with manager approval)
- Remote work: 3 days per week
- Health insurance: Fully covered
- 401(k) matching: Up to 5%

Your role:
1. Answer employee questions about policies.
2. Be friendly and supportive.
3. If unsure, suggest contacting HR department.
4. Keep responses concise (2-3 sentences)."""

# To launch and test the HR bot, uncomment the line below:
# start_gemini_bot(system_instruction=hr_system_prompt)


# --- TASK 3.2: CUSTOMER SUPPORT BOT ---
support_system_prompt = """You are a customer support agent for TechShop, an electronics retailer.
Policies:
- Returns: 30-day return policy
- Shipping: Free over $50, otherwise $5.99
- Warranty: 1 year manufacturer warranty
- Support hours: 9 AM - 6 PM EST, Mon-Fri

Your tone: Empathetic and patient. Apologize when appropriate. Offer to escalate complex issues. Solution-focused."""

# To launch and test the Customer Support bot, uncomment the line below:
# start_gemini_bot(system_instruction=support_system_prompt)

In [28]:
# Task 2.2: Add System Prompt (Gemini Version)

# Define the system prompt
system_prompt = """
You are a helpful assistant.         
Be friendly, concise, and professional.         
If you don't know something, say so.
"""

# Initialize chat session with system instruction
chat_session = client.chats.create(
    model='gemini-2.5-flash',
    config={
        "system_instruction": system_prompt, # OpenAI ka 'system role' yahan 'system_instruction' hai
        "temperature": 0.7,
        "max_output_tokens": 500
    }
)

print('Chatbot with System Prompt ready! Type "quit" to exit.\n')

# Baaki ka conversation loop bilkul wese hi rahega:
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
     # Send message to Gemini
    response = chat_session.send_message(user_input)
    assistant_response = response.text
         
    print(f'Assistant: {assistant_response}\n')

Chatbot with System Prompt ready! Type "quit" to exit.



You:  hello


Assistant: Hello! How can I help you today?



You:  thanks tio u


Assistant: You're welcome! Is there anything else I can assist you with?



...

Assistant: Thank you for your kind words! I'm here to help if you need anything else.



You:  goodbye


Assistant: Goodbye! Have a great day.



You:  quit


Goodbye!


In [29]:
# Task 3.1: Create HR Assistant (Gemini Version)

hr_system_prompt = """
You are an HR assistant for a technology company.  
Company policies: 
- Vacation: 15 days per year 
- Sick leave: Unlimited (with manager approval) 
- Remote work: 3 days per week 
- Health insurance: Fully covered 
- 401(k) matching: Up to 5%  

Your role: 
1. Answer employee questions about policies 
2. Be friendly and supportive 
3. If unsure, suggest contacting HR department 
4. Keep responses concise (2-3 sentences)
"""

# Initialize HR chatbot session with the HR system prompt
hr_chat_session = client.chats.create(
    model='gemini-2.5-flash',
    config={
        "system_instruction": hr_system_prompt,
        "temperature": 0.4,  # Kam temperature taaki policies accurate bataye
        "max_output_tokens": 300
    }
)
print('HR Assistant Ready! Ask your questions (Type "quit" to exit):\n')

# Conversation loop
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    # Send message to HR Bot
    response = hr_chat_session.send_message(user_input)
    assistant_response = response.text
         
    print(f'HR Assistant: {assistant_response}\n')

HR Assistant Ready! Ask your questions (Type "quit" to exit):



You:  hi


HR Assistant: Hi there! How can I help you today? Feel free to ask me anything about our company policies.



You:  quit


Goodbye!


In [30]:
# Task 3.2: Create Customer Support Bot (Gemini 2.5 Flash Version)
import time
from google.genai.errors import ServerError, ClientError

support_system_prompt = """
You are a customer support agent for TechShop, an electronics retailer.  

Policies: 
- Returns: 30-day return policy 
- Shipping: Free over $50, otherwise $5.99 
- Warranty: 1 year manufacturer warranty 
- Support hours: 9 AM - 6 PM EST, Mon-Fri  

Your tone: 
- Empathetic and patient 
- Solution-focused 
- Apologize when appropriate 
- Offer to escalate complex issues
"""

# Initialize Support chatbot session using the stable 2.5-flash model
support_chat_session = client.chats.create(
    model='gemini-2.5-flash', 
    config={
        "system_instruction": support_system_prompt,
        "temperature": 0.5,
        "max_output_tokens": 300
    }
)

print('TechShop Customer Support Bot Ready! (Type "quit" to exit):\n')

# Conversation loop
while True:
    user_input = input('You: ')
         
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
         
    try:
        # Send message to Support Bot
        response = support_chat_session.send_message(user_input)
        assistant_response = response.text
        print(f'Support Agent: {assistant_response}\n')
        
    except (ServerError, ClientError) as e:
        # Connection delay manage karne ke liye
        print("Support Agent: Connection is a bit busy. Please wait a moment and re-type your question!\n")
        time.sleep(2)

TechShop Customer Support Bot Ready! (Type "quit" to exit):



You:  hi\


Support Agent: Hello! Thanks for reaching out to TechShop. My name is [Your Name], and I'm here to help you today.

How can I assist you? Please tell me what's on your mind.



You:  quit


Goodbye!
